In [ ]:
# CREACION DE DATOS SIMULADOS

import pandas as pd
import numpy as np

# Semilla para obtener siempre los mismos resultados
np.random.seed(42)

# Crear 20 días laborales (4 semanas)
fechas = pd.bdate_range(
    start="2026-10-05",
    periods=20
)

tipos_prenda = [
    "Camiseta",
    "Pantaloneta",
    "Sudadera",
    "Uniforme"
]

datos = []

for fecha in fechas:

    # Producción diaria aproximada
    programadas = np.random.randint(125, 151)

    # Simular prendas pendientes
    pendientes = np.random.randint(0, 16)
    producidas = programadas - pendientes

    # Simular prendas que necesitan corrección
    defectuosas = np.random.randint(0, 7)

    datos.append({
        "fecha": fecha,
        "tipo_prenda": np.random.choice(tipos_prenda),
        "cantidad_programada": programadas,
        "cantidad_producida": producidas,
        "prendas_defectuosas": defectuosas,
        "tiempo_horas": 8,
        "operarias": 5
    })

# Convertir los datos en un DataFrame
df = pd.DataFrame(datos)

# Visualizar los primeros registros
df.head()

# CALCULAR INDICADORES GENERALES

total_programadas = df["cantidad_programada"].sum()

total_producidas = df["cantidad_producida"].sum()

total_defectuosas = df["prendas_defectuosas"].sum()

cumplimiento = (
    total_producidas / total_programadas
) * 100

print("RESUMEN DE PRODUCCIÓN")
print("----------------------------")
print("Prendas programadas:", total_programadas)
print("Prendas producidas:", total_producidas)
print("Prendas defectuosas:", total_defectuosas)
print(f"Cumplimiento: {cumplimiento:.2f}%")

# GUARDAR BASE DE DATOS EN CSV

df.to_csv(
    "produccion_textil.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo CSV creado correctamente.")

from google.colab import files

files.download("produccion_textil.csv")

In [ ]:
# Cargar la base de datos
df = pd.read_csv("produccion_textil.csv")

# Mostrar las primeras cinco filas
df.head()

from google.colab import files

archivos = files.upload()

df = pd.read_csv("produccion_textil.csv")
df.head()

# Dimensiones del dataset
print("Filas y columnas:", df.shape)

# Información de cada columna
df.info()

# Resumen estadístico
display(df.describe())

# Contar datos faltantes por columna
print(df.isnull().sum())

# Contar filas completamente duplicadas
duplicados = df.duplicated().sum()

print("Registros duplicados:", duplicados)

# Crear una copia de trabajo
df_limpio = df.copy()

# Eliminar filas exactamente duplicadas
df_limpio = df_limpio.drop_duplicates()

print("Registros después de limpiar:", len(df_limpio))

# Convertir la columna fecha a datetime
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["fecha"],
    errors="coerce"
)

# Comprobar la conversión
print(df_limpio.dtypes)

print(
    "Fechas inválidas:",
    df_limpio["fecha"].isna().sum()
)

# Detectar registros con cantidades incoherentes
errores = df_limpio[
    (df_limpio["cantidad_programada"] < 0) |
    (df_limpio["cantidad_producida"] < 0) |
    (df_limpio["prendas_defectuosas"] < 0) |
    (
        df_limpio["cantidad_producida"] >
        df_limpio["cantidad_programada"]
    ) |
    (
        df_limpio["prendas_defectuosas"] >
        df_limpio["cantidad_producida"]
    ) |
    (df_limpio["tiempo_horas"] <= 0) |
    (df_limpio["operarias"] <= 0)
]

print("Registros con posibles errores:", len(errores))

display(errores)

# Prendas que no requieren corrección
df_limpio["prendas_conformes"] = (
    df_limpio["cantidad_producida"] -
    df_limpio["prendas_defectuosas"]
)

# Cumplimiento diario de producción
df_limpio["cumplimiento_pct"] = (
    df_limpio["cantidad_producida"] /
    df_limpio["cantidad_programada"] * 100
)

# Porcentaje de defectos
df_limpio["defectos_pct"] = (
    df_limpio["prendas_defectuosas"] /
    df_limpio["cantidad_producida"] * 100
)

# Prendas producidas por hora-persona
df_limpio["productividad"] = (
    df_limpio["cantidad_producida"] /
    (
        df_limpio["tiempo_horas"] *
        df_limpio["operarias"]
    )
)

display(df_limpio.head())

# Identificar cada semana desde el inicio
fecha_inicial = df_limpio["fecha"].min()

df_limpio["semana"] = (
    (df_limpio["fecha"] - fecha_inicial)
    .dt.days // 7
) + 1

# Agrupar producción por semana
produccion_semanal = (
    df_limpio.groupby("semana")
    .agg(
        programadas=("cantidad_programada", "sum"),
        producidas=("cantidad_producida", "sum"),
        defectuosas=("prendas_defectuosas", "sum")
    )
    .reset_index()
)

# Calcular cumplimiento semanal
produccion_semanal["cumplimiento_pct"] = (
    produccion_semanal["producidas"] /
    produccion_semanal["programadas"] * 100
)

display(produccion_semanal)

# Configurar figura
plt.figure(figsize=(10, 5))

# Gráfica de producción programada
plt.plot(
    produccion_semanal["semana"],
    produccion_semanal["programadas"],
    marker="o",
    label="Producción programada"
)

# Gráfica de producción real
plt.plot(
    produccion_semanal["semana"],
    produccion_semanal["producidas"],
    marker="o",
    label="Producción real"
)

plt.title("Producción textil por semana")
plt.xlabel("Semana")
plt.ylabel("Cantidad de prendas")
plt.xticks(produccion_semanal["semana"])
plt.legend()
plt.grid(alpha=0.3)

plt.show()

produccion_por_prenda = (
    df_limpio.groupby("tipo_prenda")
    ["cantidad_producida"]
    .sum()
    .sort_values(ascending=False)
)

print("Producción por tipo de prenda:")
display(produccion_por_prenda)

# Ordenar registros cronológicamente
df_limpio = df_limpio.sort_values("fecha")

# Guardar el dataset procesado
df_limpio.to_csv(
    "produccion_textil_limpia.csv",
    index=False,
    encoding="utf-8-sig"
)

# Descargarlo al computador
from google.colab import files
files.download("produccion_textil_limpia.csv")

print("Dataset procesado correctamente.")

In [ ]:
# Cargar datos procesados
df = pd.read_csv(
    "produccion_textil_limpia.csv",
    parse_dates=["fecha"]
)

# Verificar estructura y contenido
print("Dimensiones:", df.shape)
display(df.head())

# Verificar que existan las columnas necesarias
columnas = [
    "fecha",
    "semana",
    "tipo_prenda",
    "cantidad_programada",
    "cantidad_producida",
    "prendas_defectuosas",
    "tiempo_horas",
    "operarias"
]

faltantes = set(columnas) - set(df.columns)

if faltantes:
    raise ValueError(
        f"Faltan estas columnas: {faltantes}"
    )

print("Dataset validado correctamente.")

from google.colab import files
files.upload()

# Calcular horas-persona por registro
df["horas_persona"] = (
    df["tiempo_horas"] * df["operarias"]
)

# Agrupar información semanal
reporte = (
    df.groupby("semana")
    .agg(
        programadas=("cantidad_programada", "sum"),
        producidas=("cantidad_producida", "sum"),
        defectuosas=("prendas_defectuosas", "sum"),
        horas_persona=("horas_persona", "sum")
    )
    .reset_index()
    .sort_values("semana")
)

# Indicadores generales
reporte["cumplimiento_pct"] = (
    reporte["producidas"] /
    reporte["programadas"] * 100
)

reporte["defectos_pct"] = (
    reporte["defectuosas"] /
    reporte["producidas"] * 100
)

reporte["productividad"] = (
    reporte["producidas"] /
    reporte["horas_persona"]
)

reporte["pendientes"] = (
    reporte["programadas"] -
    reporte["producidas"]
)

reporte["conformes"] = (
    reporte["producidas"] -
    reporte["defectuosas"]
)

display(reporte.round(2))

semana_critica = reporte.loc[
    reporte["cumplimiento_pct"].idxmin()
]

print("DIAGNÓSTICO DE CUMPLIMIENTO")
print("----------------------------")

print(
    "Semana:",
    int(semana_critica["semana"])
)

print(
    "Cumplimiento:",
    f'{semana_critica["cumplimiento_pct"]:.2f}%'
)

print(
    "Prendas pendientes:",
    int(semana_critica["pendientes"])
)

semana_defectos = reporte.loc[
    reporte["defectos_pct"].idxmax()
]

print("DIAGNÓSTICO DE CALIDAD")
print("----------------------")

print(
    "Semana con mayor tasa de defectos:",
    int(semana_defectos["semana"])
)

print(
    "Porcentaje de defectos:",
    f'{semana_defectos["defectos_pct"]:.2f}%'
)

plt.figure(figsize=(9, 5))

plt.bar(
    reporte["semana"].astype(str),
    reporte["defectos_pct"]
)

plt.title("Tasa de defectos por semana")
plt.xlabel("Semana")
plt.ylabel("Defectos (%)")
plt.grid(axis="y", alpha=0.3)

plt.show()

# Cambio porcentual frente a la semana anterior
reporte["variacion_productividad_pct"] = (
    reporte["productividad"].pct_change() * 100
)

display(
    reporte[
        [
            "semana",
            "productividad",
            "variacion_productividad_pct"
        ]
    ].round(2)
)

reporte_prendas = (
    df.groupby("tipo_prenda")
    .agg(
        producidas=("cantidad_producida", "sum"),
        defectuosas=("prendas_defectuosas", "sum")
    )
    .reset_index()
)

reporte_prendas["defectos_pct"] = (
    reporte_prendas["defectuosas"] /
    reporte_prendas["producidas"] * 100
)

display(
    reporte_prendas.sort_values(
        "defectos_pct",
        ascending=False
    ).round(2)
)

# Umbrales didácticos configurables
META_CUMPLIMIENTO = 95
MAX_DEFECTOS = 3

def diagnosticar_semana(fila):
    recomendaciones = []

    if fila["cumplimiento_pct"] < META_CUMPLIMIENTO:
        recomendaciones.append(
            "Revisar programación y causas "
            "de prendas pendientes"
        )

    if fila["defectos_pct"] > MAX_DEFECTOS:
        recomendaciones.append(
            "Investigar causas de defectos "
            "y controles de calidad"
        )

    if not recomendaciones:
        recomendaciones.append(
            "Mantener seguimiento de indicadores"
        )

    return "; ".join(recomendaciones)

reporte["recomendaciones"] = reporte.apply(
    diagnosticar_semana,
    axis=1
)

display(
    reporte[
        [
            "semana",
            "cumplimiento_pct",
            "defectos_pct",
            "recomendaciones"
        ]
    ]
)

reporte.to_csv(
    "diagnostico_produccion_semanal.csv",
    index=False,
    encoding="utf-8-sig"
)

reporte_prendas.to_csv(
    "diagnostico_por_prenda.csv",
    index=False,
    encoding="utf-8-sig"
)

from google.colab import files

files.download("diagnostico_produccion_semanal.csv")
files.download("diagnostico_por_prenda.csv")

In [ ]:
%pip install -q plotly ipywidgets

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Cargar archivos del paso 4
semanal = pd.read_csv(
    "diagnostico_produccion_semanal.csv"
)

prendas = pd.read_csv(
    "diagnostico_por_prenda.csv"
)

# Configuración visual
AZUL_OSCURO = "#14243B"
AZUL = "#2563EB"
VERDE = "#0D9488"
AMBAR = "#F59E0B"
FONDO = "#F3F6FB"

semanal = semanal.sort_values("semana").copy()

# Validar columnas
requeridas = {
    "semana", "programadas", "producidas",
    "defectuosas", "horas_persona"
}

assert requeridas.issubset(semanal.columns), (
    "Faltan columnas del reporte semanal"
)

# Calcular tasas a partir de sus totales
semanal["cumplimiento_pct"] = (
    100 * semanal["producidas"] /
    semanal["programadas"]
)

semanal["defectos_pct"] = (
    100 * semanal["defectuosas"] /
    semanal["producidas"]
)

semanal["productividad"] = (
    semanal["producidas"] /
    semanal["horas_persona"]
)

print("Datos listos para visualizar.")

def calcular_indicadores(datos):
    programadas = datos["programadas"].sum()
    producidas = datos["producidas"].sum()
    defectuosas = datos["defectuosas"].sum()
    horas = datos["horas_persona"].sum()

    return {
        "produccion": int(producidas),
        "cumplimiento": (
            100 * producidas / programadas
            if programadas else 0
        ),
        "defectos": (
            100 * defectuosas / producidas
            if producidas else 0
        ),
        "productividad": (
            producidas / horas
            if horas else 0
        )
    }


def crear_tarjetas(k):
    indicadores = [
        ("Prendas producidas",
         f"{k['produccion']:,}", AZUL),

        ("Cumplimiento",
         f"{k['cumplimiento']:.1f}%", VERDE),

        ("Tasa de defectos",
         f"{k['defectos']:.1f}%", AMBAR),

        ("Productividad",
         f"{k['productividad']:.2f}", AZUL_OSCURO)
    ]

    tarjetas = ""

    for titulo, valor, color in indicadores:
        tarjetas += f"""
        <div style="
            background:white;
            padding:22px;
            border-radius:14px;
            border:1px solid #e5eaf0;
            box-shadow:0 3px 12px #14243b0b;
            flex:1;
            min-width:170px;
        ">
            <div style="
                color:#64748b;
                font-size:13px;
            ">{titulo}</div>

            <div style="
                color:{color};
                font-size:30px;
                font-weight:750;
                margin-top:8px;
            ">{valor}</div>
        </div>
        """

    return f"""
    <div style="
        display:flex;
        flex-wrap:wrap;
        gap:14px;
        font-family:Arial,sans-serif;
    ">
        {tarjetas}
    </div>
    """

def crear_graficos(datos):

    # GRÁFICO 1: Producción
    fig1 = go.Figure()

    fig1.add_bar(
        x=datos["semana"],
        y=datos["programadas"],
        name="Programadas",
        marker_color=AZUL
    )

    fig1.add_bar(
        x=datos["semana"],
        y=datos["producidas"],
        name="Producidas",
        marker_color=VERDE
    )

    fig1.update_layout(
        title="Producción programada vs. real",
        barmode="group",
        yaxis_title="Prendas"
    )

    # GRÁFICO 2: Productividad
    fig2 = go.Figure()

    fig2.add_scatter(
        x=datos["semana"],
        y=datos["productividad"],
        mode="lines+markers",
        line=dict(color=AZUL, width=3),
        marker=dict(size=10),
        name="Productividad"
    )

    fig2.update_layout(
        title="Productividad semanal",
        yaxis_title="Prendas/hora-persona"
    )

    # GRÁFICO 3: Defectos
    fig3 = go.Figure()

    fig3.add_bar(
        x=datos["semana"],
        y=datos["defectos_pct"],
        marker_color=AMBAR,
        name="Tasa de defectos"
    )

    fig3.update_layout(
        title="Tasa de defectos por semana",
        yaxis_title="Defectos (%)"
    )

    # Aplicar diseño consistente
    for fig in [fig1, fig2, fig3]:

        fig.update_layout(
            template="plotly_white",
            height=360,
            paper_bgcolor="white",
            plot_bgcolor="white",
            font=dict(
                family="Arial",
                color=AZUL_OSCURO
            ),
            title_font=dict(size=17),
            margin=dict(
                l=45, r=25, t=65, b=45
            ),
            hovermode="x unified"
        )

        fig.update_xaxes(
            title_text="Semana",
            type="category"
        )

        fig.update_yaxes(
            rangemode="tozero",
            gridcolor="#e8edf3"
        )

    return [fig1, fig2, fig3]

# Opciones del filtro
opciones = [("Todas las semanas", "Todas")]

opciones += [
    (f"Semana {int(s)}", int(s))
    for s in semanal["semana"].unique()
]

filtro = widgets.Dropdown(
    options=opciones,
    value="Todas",
    description="Periodo:",
    style={"description_width": "initial"}
)

salida_dashboard = widgets.Output()


def mostrar_dashboard(cambio=None):

    with salida_dashboard:
        clear_output(wait=True)

        if filtro.value == "Todas":
            datos = semanal.copy()
            periodo = "Todas las semanas"
        else:
            datos = semanal[
                semanal["semana"] == filtro.value
            ].copy()

            periodo = f"Semana {filtro.value}"

        # Encabezado
        display(HTML(f"""
        <div style="
            background:{AZUL_OSCURO};
            color:white;
            padding:26px;
            border-radius:14px;
            font-family:Arial,sans-serif;
            margin-bottom:18px;
        ">
            <div style="
                font-size:12px;
                color:#a9c4e6;
                letter-spacing:1px;
            ">
                OPERACIONES / ANALÍTICA
            </div>

            <h1 style="
                color:white;
                margin:12px 0 8px;
                font-size:27px;
            ">
                Producción Textil
            </h1>

            <div style="color:#d6e3f4;">
                {periodo} | Proyecto académico
            </div>
        </div>
        """))

        # Tarjetas
        indicadores = calcular_indicadores(datos)

        display(HTML(
            crear_tarjetas(indicadores)
        ))

        # Gráficos
        for figura in crear_graficos(datos):
            figura.show(
                config={"displaylogo": False}
            )


# Conectar el filtro
filtro.observe(
    mostrar_dashboard,
    names="value"
)

display(filtro)
display(salida_dashboard)

mostrar_dashboard()

fig_prendas = px.bar(
    prendas.sort_values(
        "producidas",
        ascending=True
    ),
    x="producidas",
    y="tipo_prenda",
    orientation="h",
    text="producidas",
    color_discrete_sequence=[VERDE],
    title="Producción acumulada por tipo de prenda"
)

fig_prendas.update_layout(
    template="plotly_white",
    height=400,
    xaxis_title="Prendas producidas",
    yaxis_title="Tipo de prenda",
    font=dict(
        family="Arial",
        color=AZUL_OSCURO
    ),
    showlegend=False
)

fig_prendas.show()

import plotly.io as pio
from google.colab import files

# Indicadores del periodo completo
k = calcular_indicadores(semanal)

# Construir figuras
figuras = crear_graficos(semanal)
figuras.append(fig_prendas)

# Convertir gráficos a HTML
bloques = []

for i, figura in enumerate(figuras):
    html_figura = pio.to_html(
        figura,
        full_html=False,
        include_plotlyjs=(i == 0)
    )

    bloques.append(html_figura)

# Construir reporte HTML
contenido = f"""
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport"
      content="width=device-width, initial-scale=1">
<title>Dashboard de Producción Textil</title>
<style>
body {{
    font-family:Arial,sans-serif;
    background:#f3f6fb;
    margin:0;
    padding:24px;
    color:#14243b;
}}
.contenedor {{
    max-width:1100px;
    margin:auto;
}}
.encabezado {{
    background:#14243b;
    color:white;
    padding:30px;
    border-radius:16px;
}}
.tarjetas {{
    margin:22px 0;
}}
.grafico {{
    background:white;
    padding:14px;
    margin-bottom:18px;
    border-radius:14px;
}}
.nota {{
    color:#64748b;
    font-size:13px;
    margin:25px 0;
}}
</style>
</head>
<body>
<div class="contenedor">

<div class="encabezado">
<h1>Dashboard de Producción Textil</h1>
<p>Análisis de producción, calidad y productividad</p>
<p>Proyecto académico | Datos simulados</p>
</div>

<div class="tarjetas">
{crear_tarjetas(k)}
</div>

{''.join(
    '<div class="grafico">' + g + '</div>'
    for g in bloques
)}

<p class="nota">
Desarrollado con Python, pandas y Plotly.
Los resultados corresponden a una simulación
académica y no representan datos empresariales reales.
</p>

</div>
</body>
</html>
"""

nombre_archivo = "Dashboard_Produccion_Textil.html"

with open(
    nombre_archivo,
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write(contenido)

files.download(nombre_archivo)

print("Dashboard HTML exportado correctamente.")